# Tactical Pattern Recognition

**Chapter 5: Advanced Classification Methods - EXTRA**

## Learning Objectives

- Classify team playing styles using aggregated statistics
- Use decision trees for interpretable tactical rules
- Identify direct vs possession-based teams
- Extract actionable coaching insights

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

## The Problem: Classifying Playing Styles

**Question:** Can we automatically classify a team's playing style based on match statistics?

**Playing Styles:**
- **Direct**: Long passes, quick transitions, fewer touches
- **Possession**: Short passes, patient build-up, high pass completion
- **Counter-Attack**: Fast breaks, vertical passes, exploiting space

**Why This Matters:**
- Scout opponents
- Prepare tactical game plans
- Identify team identity

## Create Synthetic Team Data

For demonstration, we'll create synthetic team statistics. In practice, you'd aggregate real event data.

In [ ]:
# Create synthetic team match data
np.random.seed(42)
n_matches = 150

# Generate data for three playing styles
def generate_team_data(style, n_samples):
    if style == 'Direct':
        return pd.DataFrame({
            'avg_pass_length': np.random.normal(28, 4, n_samples),
            'pass_completion': np.random.normal(72, 5, n_samples),
            'time_per_possession': np.random.normal(7, 1.5, n_samples),
            'forward_passes_pct': np.random.normal(45, 5, n_samples),
            'style': ['Direct'] * n_samples
        })
    elif style == 'Possession':
        return pd.DataFrame({
            'avg_pass_length': np.random.normal(18, 3, n_samples),
            'pass_completion': np.random.normal(88, 3, n_samples),
            'time_per_possession': np.random.normal(15, 2, n_samples),
            'forward_passes_pct': np.random.normal(30, 4, n_samples),
            'style': ['Possession'] * n_samples
        })
    else:  # Counter-Attack
        return pd.DataFrame({
            'avg_pass_length': np.random.normal(24, 3, n_samples),
            'pass_completion': np.random.normal(78, 4, n_samples),
            'time_per_possession': np.random.normal(6, 1, n_samples),
            'forward_passes_pct': np.random.normal(55, 5, n_samples),
            'style': ['Counter-Attack'] * n_samples
        })

# Generate data
direct_data = generate_team_data('Direct', 50)
possession_data = generate_team_data('Possession', 50)
counter_data = generate_team_data('Counter-Attack', 50)

# Combine
team_data = pd.concat([direct_data, possession_data, counter_data], ignore_index=True)

print(f"Total matches: {len(team_data)}")
print(f"\nStyle distribution:")
print(team_data['style'].value_counts())
print(f"\nSample data:")
print(team_data.head())

## Visualize Playing Styles

In [ ]:
# Scatter plot: Pass length vs Completion rate
plt.figure(figsize=(12, 6))
for style in ['Direct', 'Possession', 'Counter-Attack']:
    subset = team_data[team_data['style'] == style]
    plt.scatter(subset['avg_pass_length'], subset['pass_completion'], 
                label=style, alpha=0.6, s=80)

plt.xlabel('Average Pass Length (meters)', fontsize=12)
plt.ylabel('Pass Completion Rate (%)', fontsize=12)
plt.title('Team Playing Styles: Pass Length vs Completion Rate', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Prepare Data

In [ ]:
# Features and target
feature_cols = ['avg_pass_length', 'pass_completion', 'time_per_possession', 'forward_passes_pct']
X = team_data[feature_cols]
y = team_data['style']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training set: {len(X_train)} matches")
print(f"Test set: {len(X_test)} matches")

## Train Decision Tree

We'll use a shallow tree for interpretability.

In [ ]:
# Train decision tree
tree_clf = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_clf.fit(X_train, y_train)

# Predictions
y_pred = tree_clf.predict(X_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred))

## Visualize the Decision Tree

In [ ]:
# Visualize tree
plt.figure(figsize=(20, 10))
plot_tree(tree_clf, 
          feature_names=feature_cols, 
          class_names=['Counter-Attack', 'Direct', 'Possession'],
          filled=True,
          fontsize=10,
          rounded=True)
plt.title('Decision Tree for Tactical Pattern Recognition', fontsize=16)
plt.tight_layout()
plt.show()

## Extract Tactical Rules

The tree gives us interpretable if-then rules!

In [ ]:
# Example rules extracted from the tree
print("Tactical Classification Rules:\n")
print("Rule 1: If pass_completion > 82% and time_per_possession > 10s")
print("        → Classify as POSSESSION\n")
print("Rule 2: If avg_pass_length > 25m and forward_passes_pct > 50%")
print("        → Classify as COUNTER-ATTACK\n")
print("Rule 3: If avg_pass_length > 25m and forward_passes_pct ≤ 50%")
print("        → Classify as DIRECT\n")
print("\nThese rules can be shared directly with coaching staff!")

## Feature Importance

In [ ]:
# Feature importance
importances = tree_clf.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("Feature Importances:")
print(feature_importance_df.to_string(index=False))

plt.figure(figsize=(10, 6))
plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'])
plt.xlabel('Importance')
plt.title('Feature Importance for Tactical Pattern Recognition')
plt.tight_layout()
plt.show()

## Coaching Insights

**From the model, coaches can:**

1. **Identify opponent style** before matches
   - Prepare specific tactics
   - Adjust formation and pressing strategy

2. **Monitor own team's style**
   - Ensure players execute game plan
   - Detect tactical drift over season

3. **Scout players**
   - Find players suited to desired style
   - Identify tactical versatility

**Example:** If facing a possession team (high completion, long possessions), prepare high-press tactics to disrupt their rhythm.

## Summary

In this case study, we:

1. ✅ Classified team playing styles using match statistics
2. ✅ Built an interpretable decision tree
3. ✅ Extracted tactical if-then rules
4. ✅ Identified key features distinguishing styles
5. ✅ Generated actionable coaching insights

## Key Takeaways

- **Decision trees** are perfect for tactical analysis (interpretable rules)
- **Aggregated statistics** reveal team identity
- **Shallow trees** (max_depth=3-4) balance accuracy and interpretability
- Rules can be **directly communicated** to non-technical staff

## Practice Exercises

1. **Real Data**: Use actual StatsBomb match data aggregated by team
2. **More Styles**: Add "High Press" or "Low Block" categories
3. **Temporal Analysis**: Track how team style changes over a season
4. **Player-Level**: Classify individual player styles
5. **Random Forest**: Compare with ensemble methods for higher accuracy